# Text Embeddings and Visualization
## Bag of Words, TF-IDF, and Word2Vec

This notebook demonstrates three fundamental text embedding techniques:
1. **Bag of Words (BoW)** - Simple word frequency representation
2. **TF-IDF** - Term Frequency-Inverse Document Frequency weighting
3. **Word2Vec** - Dense semantic word embeddings

We'll visualize these embeddings to understand how they capture text information.

In [ ]:
# Install required packages
!pip install -q scikit-learn gensim matplotlib numpy

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

## Sample Corpus

We'll use a simple corpus of sentences about different topics to demonstrate the embeddings.

In [ ]:
# Sample documents
documents = [
    "The cat sits on the mat",
    "The dog plays in the park",
    "Cats and dogs are pets",
    "The park has many trees",
    "Machine learning is fascinating",
    "Deep learning uses neural networks",
    "Neural networks learn patterns",
    "Python is great for machine learning"
]

print("Sample Corpus:")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc}")

## 1. Bag of Words (BoW)

BoW represents text as a vector of word counts, ignoring grammar and word order.

In [ ]:
# Create Bag of Words representation
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(documents)

# Get vocabulary
vocabulary = bow_vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {list(vocabulary)}")
print(f"\nBoW Matrix shape: {bow_matrix.shape}")
print(f"(documents x unique words): {bow_matrix.shape[0]} x {bow_matrix.shape[1]}")

In [ ]:
# Display BoW for first document
import pandas as pd

print("\nBag of Words representation for first document:")
print(f"Document: '{documents[0]}'\n")

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vocabulary)
print(bow_df.iloc[0][bow_df.iloc[0] > 0].to_dict())

In [ ]:
# Visualize BoW using PCA
pca_bow = PCA(n_components=2)
bow_2d = pca_bow.fit_transform(bow_matrix.toarray())

plt.figure(figsize=(10, 6))
plt.scatter(bow_2d[:, 0], bow_2d[:, 1], s=100, alpha=0.6, c=range(len(documents)), cmap='viridis')

for i, doc in enumerate(documents):
    plt.annotate(f"Doc {i+1}", (bow_2d[i, 0], bow_2d[i, 1]), 
                fontsize=9, alpha=0.8)

plt.title('Bag of Words - PCA Visualization', fontsize=14, fontweight='bold')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Documents closer together have similar word distributions.")

## 2. TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF weighs words by their importance:
- **TF**: How often a word appears in a document
- **IDF**: How rare a word is across all documents

Formula: `TF-IDF = TF × IDF`

In [ ]:
# Create TF-IDF representation
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Display TF-IDF for first document
print("\nTF-IDF representation for first document:")
print(f"Document: '{documents[0]}'\n")

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
doc1_tfidf = tfidf_df.iloc[0][tfidf_df.iloc[0] > 0].sort_values(ascending=False)

print("Word : TF-IDF Score")
for word, score in doc1_tfidf.items():
    print(f"{word:15s}: {score:.4f}")

In [ ]:
# Visualize TF-IDF using PCA
pca_tfidf = PCA(n_components=2)
tfidf_2d = pca_tfidf.fit_transform(tfidf_matrix.toarray())

plt.figure(figsize=(10, 6))
plt.scatter(tfidf_2d[:, 0], tfidf_2d[:, 1], s=100, alpha=0.6, c=range(len(documents)), cmap='plasma')

for i, doc in enumerate(documents):
    plt.annotate(f"Doc {i+1}", (tfidf_2d[i, 0], tfidf_2d[i, 1]), 
                fontsize=9, alpha=0.8)

plt.title('TF-IDF - PCA Visualization', fontsize=14, fontweight='bold')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("TF-IDF gives more weight to distinctive words, creating better separation.")

## 3. Word2Vec

Word2Vec learns dense vector representations that capture semantic meaning.
Words with similar meanings have similar vectors.

In [ ]:
# Prepare sentences for Word2Vec (needs tokenized input)
sentences = [doc.lower().split() for doc in documents]

print("Tokenized sentences:")
for sent in sentences:
    print(sent)

In [ ]:
# Train Word2Vec model
# Parameters:
# - vector_size: dimension of word vectors
# - window: context window size
# - min_count: ignore words with frequency less than this
# - sg: 0 for CBOW, 1 for Skip-gram

w2v_model = Word2Vec(sentences=sentences, 
                     vector_size=50,  # embedding dimension
                     window=3,        # context window
                     min_count=1,     # minimum word frequency
                     sg=1,            # use Skip-gram
                     epochs=100)

print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimension: {w2v_model.wv.vector_size}")

In [ ]:
# Get word vector
word = "learning"
if word in w2v_model.wv:
    vector = w2v_model.wv[word]
    print(f"Vector for '{word}':")
    print(f"Shape: {vector.shape}")
    print(f"First 10 dimensions: {vector[:10]}")

In [ ]:
# Find similar words
print("\nMost similar words to 'learning':")
similar_words = w2v_model.wv.most_similar('learning', topn=5)
for word, score in similar_words:
    print(f"{word:15s}: {score:.4f}")

In [ ]:
# Visualize Word2Vec embeddings using t-SNE
words = list(w2v_model.wv.index_to_key)
word_vectors = np.array([w2v_model.wv[word] for word in words])

# Use t-SNE for dimensionality reduction (better for visualization)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(words)-1))
word_vectors_2d = tsne.fit_transform(word_vectors)

plt.figure(figsize=(12, 8))
plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], s=100, alpha=0.6, c='steelblue')

for i, word in enumerate(words):
    plt.annotate(word, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]),
                fontsize=10, alpha=0.8, fontweight='bold')

plt.title('Word2Vec Embeddings - t-SNE Visualization', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Words with similar meanings should cluster together in the visualization.")

## Comparison of Methods

Let's create a side-by-side comparison of all three methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# BoW
axes[0].scatter(bow_2d[:, 0], bow_2d[:, 1], s=100, alpha=0.6, c=range(len(documents)), cmap='viridis')
for i in range(len(documents)):
    axes[0].annotate(f"D{i+1}", (bow_2d[i, 0], bow_2d[i, 1]), fontsize=9)
axes[0].set_title('Bag of Words', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3)

# TF-IDF
axes[1].scatter(tfidf_2d[:, 0], tfidf_2d[:, 1], s=100, alpha=0.6, c=range(len(documents)), cmap='plasma')
for i in range(len(documents)):
    axes[1].annotate(f"D{i+1}", (tfidf_2d[i, 0], tfidf_2d[i, 1]), fontsize=9)
axes[1].set_title('TF-IDF', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Word2Vec
axes[2].scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], s=100, alpha=0.6, c='steelblue')
for i, word in enumerate(words):
    axes[2].annotate(word, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]), fontsize=8)
axes[2].set_title('Word2Vec', fontweight='bold', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

### Bag of Words (BoW)
- **Pros**: Simple, interpretable, fast
- **Cons**: Ignores word order, semantic meaning, creates sparse high-dimensional vectors
- **Use case**: Document classification, simple text analysis

### TF-IDF
- **Pros**: Weights important words, reduces impact of common words
- **Cons**: Still sparse, ignores semantics and word order
- **Use case**: Information retrieval, document ranking, keyword extraction

### Word2Vec
- **Pros**: Captures semantic meaning, dense vectors, word relationships
- **Cons**: Requires more training data, computationally intensive
- **Use case**: Semantic similarity, word analogies, transfer learning

## Exercise for Students

Try the following:
1. Add your own documents to the corpus and see how the visualizations change
2. Experiment with Word2Vec parameters (window size, vector dimension)
3. Try finding word analogies with Word2Vec (e.g., king - man + woman = queen)
4. Compare BoW and TF-IDF for document similarity tasks